# Titanic Survival Prediction: Data Preprocessing and Logistic Regression Analysis

**Objective:** Investigate the impact of data cleaning and preprocessing on binary classification performance using the Titanic passenger dataset. All visualizations are produced exclusively with Matplotlib.

---
## 1. Data Acquisition and Initial Inspection

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("train.csv")

print(df.head())
print(df.info())
print(df.describe())

The dataset contains 891 records across 12 features. A mix of numerical (`Age`, `Fare`, `SibSp`, `Parch`) and categorical (`Sex`, `Embarked`, `Cabin`) variables is present. The target variable is `Survived` (binary: 0/1).

---
## 2. Missing Value Analysis

In [ ]:
print(df.isnull().sum())

**Findings:**
- `Age`: approximately 177 missing entries (19.9%)
- `Cabin`: approximately 687 missing entries (77.1%)
- `Embarked`: 2 missing entries (0.2%)

Each column requires a different imputation strategy based on the proportion and nature of missingness.

---
## 3. Duplicate Record Check

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.drop_duplicates(inplace=True)
print("Shape after deduplication:", df.shape)

---
## 4. Exploratory Data Analysis (Matplotlib)

### 4.1 Target Variable Distribution

In [ ]:
survival_counts = df['Survived'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(survival_counts.index.astype(str), survival_counts.values,
       color=['#c0392b', '#27ae60'], edgecolor='black', width=0.5)
ax.set_title('Survival Distribution', fontsize=13)
ax.set_xlabel('Survived (0 = No, 1 = Yes)')
ax.set_ylabel('Count')
ax.set_xticks([0, 1])
ax.set_xticklabels(['0 (Perished)', '1 (Survived)'])
plt.tight_layout()
plt.show()

The target classes are imbalanced: approximately 61.6% of passengers perished versus 38.4% who survived. This class imbalance is noted but not addressed in the current scope, as the objective is to compare preprocessing effects rather than optimize for minority-class recall.

### 4.2 Age Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df['Age'].dropna(), bins=20, color='#2980b9', edgecolor='black', alpha=0.85)
ax.set_title('Age Distribution', fontsize=13)
ax.set_xlabel('Age (years)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

The age distribution is moderately right-skewed with a concentration between 20 and 40 years. The presence of skewness justifies using the median rather than the mean for imputation, as the median is robust to asymmetric distributions.

### 4.3 Fare Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df['Fare'], bins=20, color='#d35400', edgecolor='black', alpha=0.85)
ax.set_title('Fare Distribution', fontsize=13)
ax.set_xlabel('Fare (currency units)')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

The fare distribution exhibits heavy right-skewness. The majority of passengers paid under 50 currency units, while a small number of first-class fares exceed 200. This confirms the presence of outliers that warrant treatment.

### 4.4 Fare Boxplot (Outlier Visualization)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
bp = ax.boxplot(df['Fare'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#f0c040', color='black'),
                medianprops=dict(color='#c0392b', linewidth=2),
                whiskerprops=dict(color='black'),
                capprops=dict(color='black'))
ax.set_title('Fare - Boxplot', fontsize=13)
ax.set_ylabel('Fare')
plt.tight_layout()
plt.show()

The boxplot confirms numerous data points beyond the upper whisker, indicating statistical outliers. The median fare is approximately 14.45, with the interquartile range spanning roughly 7.9 to 31.0.

---
## 5. Missing Value Treatment

### 5.1 Age - Median Imputation

In [ ]:
median_age = df['Age'].median()
print(f"Imputation value (median): {median_age}")

df['Age'] = df['Age'].fillna(median_age)
print(f"Remaining nulls in Age: {df['Age'].isnull().sum()}")

### 5.2 Embarked - Mode Imputation

In [ ]:
mode_embarked = df['Embarked'].mode()[0]
print(f"Imputation value (mode): {mode_embarked}")

df['Embarked'] = df['Embarked'].fillna(mode_embarked)
print(f"Remaining nulls in Embarked: {df['Embarked'].isnull().sum()}")

### 5.3 Cabin - Column Removal

In [ ]:
df.drop('Cabin', axis=1, inplace=True)
print("Remaining columns:", df.columns.tolist())

**Rationale:** With approximately 77% of values missing, any imputation strategy for `Cabin` would be unreliable and likely introduce noise rather than signal.

---
## 6. Outlier Treatment - Interquartile Range (IQR) Method

In [ ]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}")
print(f"Acceptable range: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Records before filtering: {len(df)}")

df = df[(df['Fare'] >= lower_bound) & (df['Fare'] <= upper_bound)]

print(f"Records after filtering:  {len(df)}")

The IQR method is a non-parametric approach that does not assume a specific distribution. Values outside the 1.5 * IQR boundary from Q1 and Q3 are classified as outliers and removed. This primarily affects high-fare first-class tickets.

---
## 7. Feature Encoding

In [ ]:
# Label encoding for binary categorical variable
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# One-hot encoding for multi-class categorical variable
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

# Remove non-predictive identifier columns
df.drop(['Name', 'Ticket', 'PassengerId'], axis=1, inplace=True, errors='ignore')

print(df.head())
print("\nColumn dtypes:\n", df.dtypes)

`drop_first=True` is applied during one-hot encoding to avoid multicollinearity (the dummy variable trap). The reference category (`Embarked_C`) is implicitly encoded when both `Embarked_Q` and `Embarked_S` are zero.

---
## 8. Baseline Model - Minimally Preprocessed Data

To establish a baseline, a separate copy of the raw data is prepared with only the minimal transformations required to run the classifier. Missing rows are dropped rather than imputed, and no outlier treatment is applied.

In [ ]:
df_raw = pd.read_csv("train.csv")

df_raw = df_raw.drop(['Cabin', 'Name', 'Ticket'], axis=1)
df_raw['Sex'] = df_raw['Sex'].map({'male': 0, 'female': 1})
df_raw['Embarked'] = df_raw['Embarked'].fillna(df_raw['Embarked'].mode()[0])
df_raw = pd.get_dummies(df_raw, columns=['Embarked'], drop_first=True)
df_raw = df_raw.dropna()  # Drop rows with remaining NaN values

print("Baseline dataset shape:", df_raw.shape)
print(df_raw.head())

---
## 9. Logistic Regression - Baseline (Raw Data)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df_raw.drop('Survived', axis=1)
y = df_raw['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_raw = LogisticRegression(max_iter=1000)
model_raw.fit(X_train, y_train)

pred_raw = model_raw.predict(X_test)

acc_raw = accuracy_score(y_test, pred_raw)
print(f"Baseline Accuracy (Raw Data): {acc_raw * 100:.2f}%")

---
## 10. Logistic Regression - Cleaned Data

In [ ]:
X = df.drop('Survived', axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_clean = LogisticRegression(max_iter=1000)
model_clean.fit(X_train, y_train)

pred_clean = model_clean.predict(X_test)

acc_clean = accuracy_score(y_test, pred_clean)
print(f"Cleaned Data Accuracy: {acc_clean * 100:.2f}%")

---
## 11. Comparative Analysis

In [ ]:
print("=" * 50)
print("  MODEL ACCURACY COMPARISON")
print("=" * 50)
print(f"  Baseline (Raw Data)  : {acc_raw * 100:.2f}%")
print(f"  Cleaned Data         : {acc_clean * 100:.2f}%")
print(f"  Difference           : {(acc_clean - acc_raw) * 100:+.2f} pp")
print("=" * 50)

if acc_clean > acc_raw:
    print("\nConclusion: Data cleaning improved classification accuracy.")
elif acc_clean == acc_raw:
    print("\nConclusion: No measurable difference in accuracy between the two pipelines.")
else:
    print("\nConclusion: Cleaning reduced accuracy, likely due to data loss from outlier removal or distribution shift.")

In [ ]:
# Visual comparison
fig, ax = plt.subplots(figsize=(6, 4))

labels = ['Baseline\n(Raw Data)', 'Cleaned\nData']
accuracies = [acc_raw * 100, acc_clean * 100]
colors = ['#7f8c8d', '#2c3e50']

bars = ax.bar(labels, accuracies, color=colors, edgecolor='black', width=0.45)
ax.set_title('Classification Accuracy: Baseline vs. Cleaned Pipeline', fontsize=12)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)

for bar, val in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 1.5,
            f"{val:.2f}%", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

---
## Summary of Findings

| Aspect | Observation | Action |
|--------|-------------|--------|
| **Age** | 19.9% missing; moderately right-skewed | Median imputation |
| **Embarked** | 2 missing values; categorical | Mode imputation |
| **Cabin** | 77.1% missing | Column dropped |
| **Fare** | Heavy right-skew with extreme outliers | Outliers removed via IQR |
| **Duplicates** | Checked for duplicate records | Removed if present |
| **Sex** | Binary categorical | Label encoded (0/1) |
| **Embarked** | Multi-class categorical | One-hot encoded (drop_first) |
| **Model** | Logistic Regression (baseline vs. cleaned) | Cleaning generally improves accuracy |